In [1]:
# cell 1 — set up imports & paths
import sys
from pathlib import Path
import json
import pandas as pd

# -------------- 修改为你本地的项目根路径 --------------
PROJECT_ROOT = Path("/Users/shiyiliu/Desktop/Latent_Space_Vis").resolve()
# --------------------------------------------------------

# 插入 scripts/GNNGMMAnalyzer 到 sys.path（使 embedding_pipeline_core 可导入）
GNNGMM_DIR = PROJECT_ROOT / "scripts" / "GNNGMMAnalyzer"
if str(GNNGMM_DIR) not in sys.path:
    sys.path.insert(0, str(GNNGMM_DIR))

# 如果 backend 里有 util 也需要，可以加入 backend 目录
BACKEND_DIR = PROJECT_ROOT / "backend"
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("GNNGMM_DIR on sys.path:", GNNGMM_DIR.exists())
print("BACKEND_DIR on sys.path:", BACKEND_DIR.exists())

# 现在导入后端模块（和 backend/app.py 使用的同样函数）
from embedding_pipeline_core import load_embeddings_from_files
from embedding_pipeline import EmbeddingPipeline

print("Modules imported OK")

PROJECT_ROOT: /Users/shiyiliu/Desktop/Latent_Space_Vis
GNNGMM_DIR on sys.path: True
BACKEND_DIR on sys.path: True
Modules imported OK


In [3]:
# cell 2 — load embeddings and run pipeline
import torch   # 若 load_embeddings_from_files 内使用 torch
from pathlib import Path

# Embeddings 目录 —— 改为你本地路径（最好用绝对路径）
EMB_DIR = PROJECT_ROOT / "backend" / "anyembeddings"   # example: /Users/.../Latent_Space_Vis/anyembeddings

# 可选：如果你想指定具体 layer 文件与名字（与你 frontend 使用的一致）
layer_files = ["emb_unique_cas.pt","output0_unique_cas.pt","output1_unique_cas.pt"]
layer_names = ["emb_unique_cas","output0_unique_cas","output1_unique_cas"]
layer_files_dict = dict(zip(layer_names, layer_files))

print("Embeddings dir exists:", EMB_DIR.exists())
if not EMB_DIR.exists():
    raise FileNotFoundError(f"Embeddings dir not found: {EMB_DIR}")

# 调用 load_embeddings_from_files
# 如果你的 load_embeddings_from_files 接口支持只传目录也可以删掉第二个参数
embeddings = load_embeddings_from_files(EMB_DIR, layer_files_dict)
# embeddings 应该是 dict: { layer_name: numpy/tensor array (n_samples, dim) }
print("Loaded layers:", list(embeddings.keys()))
for k, v in embeddings.items():
    try:
        print(k, getattr(v, "shape", (len(v),)))
    except Exception:
        print(k, type(v))

# 构造 metadata（可选）
metadata = {"note": "notebook run"}

# 创建 pipeline（参考 backend/app.py 的用法）
tmp_output_dir = PROJECT_ROOT / "frontend" / "public" / "data" / "tmp_notebook"
tmp_output_dir.mkdir(parents=True, exist_ok=True)

pipeline = EmbeddingPipeline(
    embeddings=embeddings,
    output_dir=tmp_output_dir,
    id_mapping=None,    # 如果有 id map 的 json path，可加载并传入 dict
    n_components=4,     # 你想降维成多少维
    metadata=metadata
)

# 运行 GMM + MDS（和 backend 一样）
print("Fitting GMMs...")
pipeline.fit_gmms()
print("Computing MDS / distance embedding...")
pipeline.compute_mds()

# 生成 scatter + sankey DataFrame
scatter_df = pipeline.generate_scatter_plot_data()
sankey_df = pipeline.generate_sankey_data()

print("scatter rows:", len(scatter_df))
print("sankey rows:", len(sankey_df))

# 简单查看前几行
display(scatter_df.head())
display(sankey_df.head())

Embeddings dir exists: True
  ✓ emb_unique_cas: shape (152, 128)
  ✓ output0_unique_cas: shape (152, 256)
  ✓ output1_unique_cas: shape (152, 256)
Loaded layers: ['emb_unique_cas', 'output0_unique_cas', 'output1_unique_cas']
emb_unique_cas (152, 128)
output0_unique_cas (152, 256)
output1_unique_cas (152, 256)
Using 4 GMM components
Fitting GMMs...

Fitting GMMs for all layers...
  Fitting GMM for emb_unique_cas...
  Fitting GMM for output0_unique_cas...
  Fitting GMM for output1_unique_cas...
✓ Fitted GMMs for 3 layers
Computing MDS / distance embedding...

Computing MDS positions...
✓ Computed MDS positions for 3 layers

Generating scatter plot data...
✓ Generated scatter plot data: 456 points

Generating Sankey diagram data...
✓ Generated Sankey diagram data: 304 transitions
scatter rows: 456
sankey rows: 304


,id,x,y,color_value,layer
0,0,1053.377761,1234.786399,0.0,emb_unique_cas
1,1,1557.396823,921.892890,0.0,emb_unique_cas
2,2,910.397481,-2696.177409,0.0,emb_unique_cas
3,3,3084.250371,813.262800,0.0,emb_unique_cas
4,4,1441.918031,-237.268487,0.0,emb_unique_cas


,real_id,from_layer,to_layer,from_cluster,to_cluster
0,0,emb_unique_cas,output0_unique_cas,0,3
1,1,emb_unique_cas,output0_unique_cas,0,3
2,2,emb_unique_cas,output0_unique_cas,2,3
3,3,emb_unique_cas,output0_unique_cas,1,3
4,4,emb_unique_cas,output0_unique_cas,2,3


In [4]:
# cell 3 — save results to disk for later使用或前端读取
OUT_DIR = tmp_output_dir
scatter_csv = OUT_DIR / "scatter_notebook.csv"
sankey_csv = OUT_DIR / "sankey_notebook.csv"

scatter_df.to_csv(scatter_csv, index=False)
sankey_df.to_csv(sankey_csv, index=False)

# 如果你更喜欢 json（scatter 通常是 records）
scatter_json = OUT_DIR / "scatter_notebook.json"
sankey_json = OUT_DIR / "sankey_notebook.json"
scatter_df.to_json(scatter_json, orient="records")
sankey_df.to_json(sankey_json, orient="records")

print("Saved scatter ->", scatter_csv)
print("Saved sankey  ->", sankey_csv)

Saved scatter -> /Users/shiyiliu/Desktop/Latent_Space_Vis/frontend/public/data/tmp_notebook/scatter_notebook.csv
Saved sankey  -> /Users/shiyiliu/Desktop/Latent_Space_Vis/frontend/public/data/tmp_notebook/sankey_notebook.csv


In [9]:
# 在 Jupyter Notebook 中直接复制运行此 cell
from pathlib import Path
import pandas as pd
import json
from IPython.display import IFrame, display

def render_sankey_notebook(sankey_csv_path, out_html_path=None, width="100%", height=480):
    sankey_csv_path = Path(sankey_csv_path)
    if not sankey_csv_path.exists():
        raise FileNotFoundError(f"{sankey_csv_path} not found")

    df = pd.read_csv(sankey_csv_path)
    rows = df.to_dict(orient="records")
    data_json = json.dumps(rows)

    if out_html_path is None:
        out_html_path = Path.cwd() / "sankey_notebook.html"
    else:
        out_html_path = Path(out_html_path)

    # 为避免 f-string 和 JS `${}` 冲突，使用字符串拼接
    html = (
"<!doctype html>\n"
"<html>\n<head>\n  <meta charset='utf-8' />\n  <meta name='viewport' content='width=device-width,initial-scale=1' />\n  <title>Sankey Notebook</title>\n  <style>\n    body { margin:0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial; }\n    #container { padding: 6px; }\n    svg { width:100%; height:100%; display:block; }\n    .node rect { stroke: none; }\n    .node text { font-size:11px; fill:#222; }\n  </style>\n  <!-- CDN: d3 and d3-sankey -->\n  <script src='https://cdnjs.cloudflare.com/ajax/libs/d3/7.10.0/d3.min.js'></script>\n  <script src='https://cdn.jsdelivr.net/npm/d3-sankey@0.12.3/dist/d3-sankey.min.js'></script>\n</head>\n<body>\n<div id='container'>\n  <div id='sankey' style='width:100%; height:" + str(height) + "px;'></div>\n</div>\n\n<script>\nconst rawRows = "
    + data_json +
";\n\n// buildGraphFromRows: nodes named 'layer_cluster', links aggregated by count\nfunction buildGraphFromRows(rows) {\n  const nodeNames = new Set();\n  const linkMap = new Map();\n  rows.forEach(r => {\n    const src = r.from_layer + '_' + r.from_cluster;\n    const tgt = r.to_layer + '_' + r.to_cluster;\n    nodeNames.add(src); nodeNames.add(tgt);\n    const key = src + '->' + tgt;\n    linkMap.set(key, (linkMap.get(key) || 0) + 1);\n  });\n  const nodes = Array.from(nodeNames).map(name => ({name}));\n  const nodeIndex = new Map(nodes.map((n,i) => [n.name, i]));\n  const links = Array.from(linkMap.entries()).map(([key,value]) => {\n    const parts = key.split('->');\n    return { source: nodeIndex.get(parts[0]), target: nodeIndex.get(parts[1]), value, key };\n  });\n  return {nodes, links};\n}\n\nfunction draw(containerSelector, rows, opts) {\n  const svgContainer = d3.select(containerSelector);\n  svgContainer.selectAll(\"*\").remove();\n  const width = svgContainer.node().clientWidth || 800;\n  const height = opts.height || 430;\n  const graph = buildGraphFromRows(rows);\n\n  const sankeyGen = d3.sankey()\n    .nodeWidth(10)\n    .nodePadding(7)\n    .extent([[20, 10], [width - 20, height - 10]]);\n\n  const sankeyData = sankeyGen({\n    nodes: graph.nodes.map(d=>Object.assign({}, d)),\n    links: graph.links.map(d=>Object.assign({}, d))\n  });\n\n  const svg = svgContainer.append(\"svg\").attr(\"width\", width).attr(\"height\", height);\n\n  // map real_id -> split label\n  const splitMap = new Map();\n  rows.forEach(r => splitMap.set(String(r.real_id), r.split_label || \"missing\"));\n  const splitColors = { train:\"#1f77b4\", test:\"#ff7f0e\", experimental:\"#2ca02c\", missing:\"#999999\" };\n\n  // aggregated links\n  const linkGroup = svg.append(\"g\").attr(\"class\",\"links\");\n  linkGroup.selectAll(\"path\")\n    .data(sankeyData.links)\n    .join(\"path\")\n    .attr(\"d\", d3.sankeyLinkHorizontal())\n    .attr(\"stroke\", \"#999\")\n    .attr(\"stroke-width\", d => Math.max(1, d.width || d.value))\n    .attr(\"fill\",\"none\")\n    .attr(\"opacity\", 0.4);\n\n  // build sample paths and node->sample lists\n  const samplePaths = new Map();\n  rows.forEach(r => {\n    const sid = String(r.real_id);\n    const fromNode = r.from_layer + '_' + r.from_cluster;\n    const toNode = r.to_layer + '_' + r.to_cluster;\n    if (!samplePaths.has(sid)) samplePaths.set(sid, []);\n    const p = samplePaths.get(sid);\n    if (p.length === 0 || p[p.length-1] !== fromNode) p.push(fromNode);\n    p.push(toNode);\n  });\n  const nodeSamples = new Map();\n  samplePaths.forEach((path,sid) => { path.forEach(nm => { if (!nodeSamples.has(nm)) nodeSamples.set(nm, []); nodeSamples.get(nm).push(sid); }); });\n\n  // draw stacked nodes\n  const nodeGroup = svg.append(\"g\").attr(\"class\",\"nodes\");\n  const nodeG = nodeGroup.selectAll(\"g.node\").data(sankeyData.nodes).join(\"g\").attr(\"class\",\"node\").attr(\"transform\", d => `translate(${d.x0},0)`);\n  nodeG.each(function(d){\n    const g = d3.select(this); const nodeName = d.name; const y0 = d.y0, y1 = d.y1; const total = (nodeSamples.get(nodeName) || []).length; const nodeWidth = d.x1 - d.x0;\n    if (total === 0) { g.append(\"rect\").attr(\"x\",0).attr(\"y\", y0).attr(\"width\", nodeWidth).attr(\"height\", y1-y0).attr(\"fill\",\"#ccc\"); return; }\n    const counts = { train:0, test:0, experimental:0, missing:0 };\n    (nodeSamples.get(nodeName) || []).forEach(sid => { const lab = splitMap.get(sid) || \"missing\"; counts[lab] = (counts[lab]||0) + 1; });\n    const splits = [\"train\",\"test\",\"experimental\",\"missing\"]; let curY = y0;\n    splits.forEach(sp => { const c = counts[sp] || 0; if (c<=0) return; const h = ((y1-y0) * c) / total; g.append(\"rect\").attr(\"x\",0).attr(\"y\",curY).attr(\"width\",nodeWidth).attr(\"height\",h).attr(\"fill\", splitColors[sp]); curY += h; });\n  });\n\n  // labels\n  svg.append(\"g\").selectAll(\"text\").data(sankeyData.nodes).join(\"text\").attr(\"x\", d=> (d.x0||0)-6).attr(\"y\", d=>((d.y0||0)+(d.y1||0))/2).attr(\"dy\",\"0.35em\").attr(\"text-anchor\",\"end\").text(d=>d.name);\n\n  // compute sample y positions and draw trajectories (thin lines)\n  const sampleNodeY = new Map();\n  sankeyData.nodes.forEach(n=>{ const nodeName=n.name; const samples=nodeSamples.get(nodeName)||[]; samples.sort(); const y0=n.y0||0, y1=n.y1||0, h=y1-y0; samples.forEach((sid,i)=>{ const y = samples.length===1 ? y0 + h/2 : y0 + (i+0.5) * (h / samples.length); if (!sampleNodeY.has(sid)) sampleNodeY.set(sid, new Map()); sampleNodeY.get(sid).set(nodeName, y); }); });\n  const lines = [];\n  samplePaths.forEach((path,sid)=>{ for (let i=0;i<path.length-1;i++){ const sNode=path[i], tNode=path[i+1]; const sObj = sankeyData.nodes.find(n=>n.name===sNode); const tObj = sankeyData.nodes.find(n=>n.name===tNode); if (!sObj||!tObj) continue; const x1=(sObj.x0||0)+(sObj.x1-sObj.x0)/2; const x2=(tObj.x0||0)+(tObj.x1-tObj.x0)/2; const y1 = sampleNodeY.get(sid)?.get(sNode); const y2 = sampleNodeY.get(sid)?.get(tNode); if (y1===undefined||y2===undefined) continue; lines.push({x1,y1,x2,y2,sid}); } });\n  svg.append(\"g\").attr(\"class\",\"sample-trajectories\").selectAll(\"line\").data(lines).join(\"line\").attr(\"x1\",d=>d.x1).attr(\"y1\",d=>d.y1).attr(\"x2\",d=>d.x2).attr(\"y2\",d=>d.y2).attr(\"stroke\", d => splitColors[ splitMap.get(d.sid) || \"missing\" ]).attr(\"stroke-width\",0.6).attr(\"opacity\",0.6);\n}\n\n// draw\ndraw(\"#sankey\", rawRows, {height: " + str(height) + "});\n\n</script>\n</body>\n</html>\n"
    )

    out_html_path.write_text(html, encoding="utf-8")
    display(IFrame(src=str(out_html_path), width=width, height=height+40))
    return out_html_path

In [10]:
render_sankey_notebook(sankey_csv)

PosixPath('/Users/shiyiliu/Desktop/Latent_Space_Vis/sankey_notebook.html')